### Imports + paths (auto-find your latest chunks + FAISS)

In [1]:
import os, glob
import numpy as np
import pandas as pd
import faiss

# If you're running inside /notebooks folder:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# latest chunks csv
chunk_candidates = sorted(
    glob.glob(os.path.join(DATA_DIR, "arxiv_chunks_*.csv")),
    key=os.path.getmtime
)
CHUNKS_PATH = chunk_candidates[-1] if chunk_candidates else None

# faiss index path (expected)
FAISS_DIR = os.path.join(DATA_DIR, "faiss")
FAISS_INDEX_PATH = os.path.join(FAISS_DIR, "arxiv_faiss.index")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("CHUNKS_PATH:", CHUNKS_PATH)
print("FAISS_INDEX_PATH:", FAISS_INDEX_PATH)
print("FAISS_INDEX_EXISTS:", os.path.exists(FAISS_INDEX_PATH))

PROJECT_ROOT: C:\Users\vidus\Projects\RAG-LLM-Projects\AskMyDocs-RAG-Chatbot
DATA_DIR: C:\Users\vidus\Projects\RAG-LLM-Projects\AskMyDocs-RAG-Chatbot\data\processed
CHUNKS_PATH: C:\Users\vidus\Projects\RAG-LLM-Projects\AskMyDocs-RAG-Chatbot\data\processed\arxiv_chunks_20260223_1021.csv
FAISS_INDEX_PATH: C:\Users\vidus\Projects\RAG-LLM-Projects\AskMyDocs-RAG-Chatbot\data\processed\faiss\arxiv_faiss.index
FAISS_INDEX_EXISTS: True


In [3]:
import ollama

models = ollama.list()["models"]

print("Ollama reachable ✅")
print("Installed models:", [m["model"] for m in models])

Ollama reachable ✅
Installed models: ['llama3.1:latest']


In [5]:
!pip install ollama

### Load chunks CSV into dataframe

In [7]:
df_chunks = pd.read_csv(CHUNKS_PATH)
print("df_chunks shape:", df_chunks.shape)
df_chunks.head(2)

df_chunks shape: (136, 7)


,chunk_id,doc_id,chunk_index,chunk_text,title,categories,update_date
0,704.0047__0,704.0047,0,Title: Intelligent location of simultaneously ...,Intelligent location of simultaneously active ...,cs.NE cs.AI,2009-09-29
1,704.0047__1,704.0047,1,rning from examples. Locator performance was t...,Intelligent location of simultaneously active ...,cs.NE cs.AI,2009-09-29


### Load FAISS index

In [9]:
index = faiss.read_index(FAISS_INDEX_PATH)
print("FAISS index loaded ✅")
print("ntotal vectors:", index.ntotal)

FAISS index loaded ✅
ntotal vectors: 136


### Reduce warnings and log noice

In [11]:
import warnings, logging

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

### Load embedding model (same as vector store)

In [13]:
import logging
from sentence_transformers import SentenceTransformer

logging.getLogger("transformers").setLevel(logging.ERROR)

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

print("Embedder loaded ✅", EMBED_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedder loaded ✅ sentence-transformers/all-MiniLM-L6-v2


## Load Embedding Model (Same as Vector Store)

The embedding model must match the one used to build the FAISS index.

We use:
- sentence-transformers/all-MiniLM-L6-v2
- 384-dimensional embeddings
- Normalized vectors for cosine similarity

### Retrieval function (FAISS search)

## Retrieval Layer (FAISS Semantic Search)

This function performs semantic retrieval using FAISS.

Steps:
1. Convert the user query into an embedding.
2. Perform similarity search using FAISS.
3. Retrieve top-k relevant chunks.
4. Return similarity scores and metadata.

In [15]:
def retrieve(query: str, k: int = 5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    distances, indices = index.search(q_emb, k)

    results = df_chunks.iloc[indices[0]].copy()
    results["score"] = distances[0]
    return results.reset_index(drop=True)

# quick test
retrieve("What is retrieval augmented generation?", k=5)[["score", "doc_id", "chunk_id", "title"]].head()

,score,doc_id,chunk_id,title
0,0.329056,704.1676,704.1676__2,Personalizing Image Search Results on Flickr
1,0.317295,704.1409,704.1409__0,Preconditioned Temporal Difference Learning
2,0.273139,704.1675,704.1675__0,Exploiting Social Annotation for Automatic Res...
3,0.253755,704.2644,704.2644__1,Joint universal lossy coding and identificatio...
4,0.249998,705.1161,705.1161__0,IDF revisited: A simple new derivation within ...


## Generation Layer (LLM Response - Local Ollama)

This section connects retrieval with generation.

Pipeline:
1. Retrieve top-k relevant chunks using FAISS.
2. Build structured context from retrieved chunks.
3. Send question + context to local LLM (Ollama).
4. Generate a grounded answer.

This implementation uses:
- FAISS for retrieval
- SentenceTransformer for embeddings
- Ollama (llama3.1) as the local LLM

### Step 1 — Verify Ollama Runtime

Before generating answers, we confirm:
- Ollama is running locally
- The llama3.1 model is installed and accessible

In [17]:
import ollama

models = ollama.list()["models"]

print("Ollama reachable ✅")
print("Installed models:", [m["model"] for m in models])

Ollama reachable ✅
Installed models: ['llama3.1:latest']


### Step 2 — Build Context from Retrieved Chunks

After retrieving the top-k relevant chunks,  
we combine them into a structured context block.

Why?

- LLMs need structured input
- We limit total characters to avoid overflow
- We attach metadata (title + chunk_id) for traceability

In [19]:
def build_context(retrieved_df, max_chars: int = 3500):
    parts = []
    total = 0

    for _, row in retrieved_df.iterrows():
        chunk_text = str(row.get("chunk_text", ""))
        title = str(row.get("title", ""))
        chunk_id = str(row.get("chunk_id", ""))

        block = f"[{title} | chunk {chunk_id}]\n{chunk_text}\n"

        if total + len(block) > max_chars:
            break

        parts.append(block)
        total += len(block)

    return "\n---\n".join(parts)

### Step 3 — Generate Answer Using Local LLM (Ollama)

Now we connect the structured context to the LLM.

We:
- Provide the retrieved context
- Provide the user question
- Instruct the model to answer only from context
- Prevent hallucination by forcing grounded responses

If the answer is not found in the retrieved chunks,
the model will respond:

"I don't know based on the retrieved documents."

In [21]:
OLLAMA_MODEL = "llama3.1:latest"

def generate_with_ollama(question: str, context: str):
    prompt = f"""
You are a helpful assistant.
Answer the question using ONLY the context provided.
If the answer is not in the context, say:
"I don't know based on the retrieved documents."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""".strip()

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]

### Step 4 — End-to-End RAG Pipeline

Now we combine everything:

1. Retrieve relevant chunks using FAISS
2. Build structured context
3. Generate grounded answer using Ollama

This function represents the complete
Retrieval-Augmented Generation (RAG) pipeline.

In [23]:
def ask_rag(question: str, k: int = 5, max_chars: int = 3500):
    
    # Step 1 — Retrieve
    retrieved = retrieve(question, k=k)
    
    # Step 2 — Build Context
    context = build_context(retrieved, max_chars=max_chars)
    
    # Step 3 — Generate Answer
    answer = generate_with_ollama(question, context)
    
    return {
        "question": question,
        "answer": answer,
        "retrieved_docs": retrieved[["score", "doc_id", "chunk_id", "title"]]
    }

### Step 5 — Test the Full RAG Pipeline

We now run one real user question end-to-end:

1. FAISS retrieves the most relevant chunks  
2. We build a context block  
3. Ollama generates a grounded answer using only that context  

We print:
- the final answer  
- the top retrieved chunks (so we can verify relevance)

In [25]:
# Test question (you can change this)
q = "What is retrieval augmented generation (RAG)?"

out = ask_rag(q, k=5, max_chars=3500)

print("QUESTION:\n", out["question"])
print("\nANSWER:\n", out["answer"])

print("\nTOP RETRIEVED CHUNKS:")
display(out["retrieved_docs"])

QUESTION:
 What is retrieval augmented generation (RAG)?

ANSWER:
 I don't know based on the retrieved documents.

TOP RETRIEVED CHUNKS:


,score,doc_id,chunk_id,title
0,0.277651,704.1676,704.1676__2,Personalizing Image Search Results on Flickr
1,0.239114,704.1675,704.1675__0,Exploiting Social Annotation for Automatic Res...
2,0.231468,704.1409,704.1409__0,Preconditioned Temporal Difference Learning
3,0.204757,706.0465,706.0465__2,Virtual Sensor Based Fault Detection and Class...
4,0.200540,704.2644,704.2644__1,Joint universal lossy coding and identificatio...


### Debug — Inspect Retrieved Context

Before generating an answer, we print the exact retrieved chunk text.
This helps verify whether the context actually contains the information needed to answer the question.

In [27]:
# 1) retrieve
retrieved = retrieve(q, k=5)

# 2) build context
context = build_context(retrieved, max_chars=3500)

print("===== CONTEXT SENT TO OLLAMA =====\n")
print(context[:3500])  # preview
print("\n===== END CONTEXT =====")

# optional: show the full chunk texts in a table
display(retrieved[["score", "doc_id", "chunk_id", "title", "chunk_text"]])

===== CONTEXT SENT TO OLLAMA =====

[Personalizing Image Search Results on Flickr | chunk 704.1676__2]
l are then used to personalize search results by finding images on topics that are of interest to the user.
Categories: cs.IR cs.AI cs.CY cs.DL cs.HC
Last updated: 2007-05-23

---
[Exploiting Social Annotation for Automatic Resource Discovery | chunk 704.1675__0]
Title: Exploiting Social Annotation for Automatic Resource Discovery
Abstract: Information integration applications, such as mediators or mashups, that require access to information resources currently rely on users manually discovering and integrating them in the application. Manual resource discovery is a slow process, requiring the user to sift through results obtained via keyword-based search. Although search methods have advanced to include evidence from document contents, its metadata and the contents and link structure of the referring pages, they still do not adequately cover information sources -- often called ``the 

,score,doc_id,chunk_id,title,chunk_text
0,0.277651,704.1676,704.1676__2,Personalizing Image Search Results on Flickr,l are then used to personalize search results ...
1,0.239114,704.1675,704.1675__0,Exploiting Social Annotation for Automatic Res...,Title: Exploiting Social Annotation for Automa...
2,0.231468,704.1409,704.1409__0,Preconditioned Temporal Difference Learning,Title: Preconditioned Temporal Difference Lear...
3,0.204757,706.0465,706.0465__2,Virtual Sensor Based Fault Detection and Class...,completed) would provide important information...
4,0.200540,704.2644,704.2644__1,Joint universal lossy coding and identificatio...,urce coding and identification. We also give s...


In [29]:
q = "What is preconditioned temporal difference learning?"
out = ask_rag(q, k=5, max_chars=3500)
print(out["answer"])
display(out["retrieved_docs"])

I don't know based on the retrieved documents.


,score,doc_id,chunk_id,title
0,0.828950,704.1409,704.1409__0,Preconditioned Temporal Difference Learning
1,0.395274,705.2318,705.2318__1,Statistical Mechanics of Nonlinear On-line Lea...
2,0.380851,706.1290,706.129__0,Temporal Reasoning without Transitive Tables
3,0.364661,706.1290,706.129__1,Temporal Reasoning without Transitive Tables
4,0.362213,705.2318,705.2318__0,Statistical Mechanics of Nonlinear On-line Lea...


## Debug — Inspect Context Sent to the LLM

Before generating an answer, print the exact context that will be sent to Ollama.

This helps verify:
- retrieved chunks contain meaningful text (not just titles/metadata)
- the context is long enough (not empty / truncated too early)
- the question is actually answerable from retrieved content

In [30]:
# Change this question to anything
q = "What is preconditioned temporal difference learning?"

retrieved = retrieve(q, k=5)

# IMPORTANT: check if chunk_text is present and non-empty
display(retrieved[["score", "doc_id", "chunk_id", "title", "chunk_text"]].head(5))

context = build_context(retrieved, max_chars=3500)

print("===== CONTEXT SENT TO OLLAMA (preview) =====")
print(context[:3500])
print("===== END CONTEXT =====")
print("\nContext length:", len(context))

,score,doc_id,chunk_id,title,chunk_text
0,0.828950,704.1409,704.1409__0,Preconditioned Temporal Difference Learning,Title: Preconditioned Temporal Difference Lear...
1,0.395274,705.2318,705.2318__1,Statistical Mechanics of Nonlinear On-line Lea...,learning show qualitatively different behavior...
2,0.380851,706.1290,706.129__0,Temporal Reasoning without Transitive Tables,Title: Temporal Reasoning without Transitive T...
3,0.364661,706.1290,706.129__1,Temporal Reasoning without Transitive Tables,transitive tables -- or similarly inference ru...
4,0.362213,705.2318,705.2318__0,Statistical Mechanics of Nonlinear On-line Lea...,Title: Statistical Mechanics of Nonlinear On-l...


===== CONTEXT SENT TO OLLAMA (preview) =====
[Preconditioned Temporal Difference Learning | chunk 704.1409__0]
Title: Preconditioned Temporal Difference Learning
Abstract: This paper has been withdrawn by the author. This draft is withdrawn for its poor quality in english, unfortunately produced by the author when he was just starting his science route. Look at the ICML version instead: http://icml2008.cs.helsinki.fi/papers/111.pdf
Categories: cs.LG cs.AI
Last updated: 2012-06-11

---
[Statistical Mechanics of Nonlinear On-line Learning for Ensemble Teachers | chunk 705.2318__1]
learning show qualitatively different behaviors from each other. In Hebbian learning, we can analytically obtain the solutions. In this case, the generalization error monotonically decreases. The steady value of the generalization error is independent of the learning rate. The larger the number of teachers is and the more variety the ensemble teachers have, the smaller the generalization error is. In perceptron

## Improvement — Filter Out Withdrawn or Low-Content Papers

Some arXiv entries contain:
- Withdrawn drafts
- Very short or low-information abstracts

We filter these before building context so the LLM only sees meaningful research content.

In [33]:
def retrieve(query: str, k: int = 5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    distances, indices = index.search(q_emb, k * 3)  # search more to allow filtering

    results = df_chunks.iloc[indices[0]].copy()
    results["score"] = distances[0]

    # Filter withdrawn / useless abstracts
    mask = ~results["chunk_text"].str.lower().str.contains("withdrawn", na=False)
    mask &= results["chunk_text"].str.len() > 100

    results = results[mask]

    return results.head(k).reset_index(drop=True)

## Test — Ask a Research-Oriented Question

Instead of asking a general definition, we ask about:
- main problem
- key contribution
- methodology
- experimental findings

These are typically present in abstracts.

In [35]:
q = "What problem does the top retrieved paper try to solve?"

out = ask_rag(q, k=5, max_chars=3500)

print("ANSWER:\n", out["answer"])
display(out["retrieved_docs"])

ANSWER:
 Based on the top retrieved paper "[Virtual Sensor Based Fault Detection and Classification on a Plasma Etch Reactor | chunk 706.0465__2]", the problem that it tries to solve is fault detection and classification in a plasma etch reactor.


,score,doc_id,chunk_id,title
0,0.246651,706.0465,706.0465__2,Virtual Sensor Based Fault Detection and Class...
1,0.240680,705.1161,705.1161__0,IDF revisited: A simple new derivation within ...
2,0.213960,705.2307,705.2307__1,A Study in a Hybrid Centralised-Swarm Agent Co...
3,0.200543,704.1274,704.1274__1,Parametric Learning and Monte Carlo Optimization
4,0.182200,704.3359,704.3359__1,Direct Optimization of Ranking Measures


## Research Question Bank (for Testing Retrieval)

Use these questions to test whether retrieval is returning relevant papers and whether the LLM can answer from the retrieved abstracts.

**Single-paper questions (top retrieved paper):**
- Summarize the abstract of the top retrieved paper.
- What is the main contribution of the retrieved paper?
- What methods are mentioned in the retrieved abstracts?
- What tasks or domains are discussed?
- List key terms from the retrieved papers.

**Comparison questions (top 2 papers):**
- Compare the first two retrieved papers based on their abstracts.
- Which retrieved paper relates most to artificial intelligence and why?

**Corpus/meta questions:**
- What categories (cs.LG, cs.AI, etc.) appear most often in the retrieved results?

In [37]:
# --- Citation-aware context builder ---
def build_context(retrieved_df, max_chars: int = 3500):
    """
    Builds a context block that includes citation handles so the LLM can cite sources.
    Each chunk is tagged like:
    [1] Title | chunk_id
    chunk_text...
    """
    parts = []
    total = 0

    for i, row in enumerate(retrieved_df.itertuples(index=False), start=1):
        title = str(getattr(row, "title", ""))
        chunk_id = str(getattr(row, "chunk_id", ""))
        chunk_text = str(getattr(row, "chunk_text", ""))

        block = f"[{i}] {title} | {chunk_id}\n{chunk_text}\n"
        if total + len(block) > max_chars:
            break

        parts.append(block)
        total += len(block)

    return "\n---\n".join(parts)


# --- Citation-enforced generation ---
OLLAMA_MODEL = "llama3.1:latest"

def generate_with_ollama(question: str, context: str):
    prompt = f"""
You are a careful research assistant.

You MUST answer using ONLY the provided context.
You MUST include citations in square brackets using the source numbers, like: [1] or [1][3].
If the answer is not present in the context, say exactly:
"I don't know based on the retrieved documents."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER (with citations):
""".strip()

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]


# --- End-to-end function (uses updated build_context + generation) ---
def ask_rag(question: str, k: int = 5, max_chars: int = 3500):
    retrieved = retrieve(question, k=k)
    context = build_context(retrieved, max_chars=max_chars)
    answer = generate_with_ollama(question, context)

    return {
        "question": question,
        "answer": answer,
        "retrieved_docs": retrieved[["score", "doc_id", "chunk_id", "title"]]
    }

## Quick Test (with citations)

We run a research-style question and verify:
- The answer is grounded in the retrieved abstracts
- The response includes citations like [1] or [2]

In [39]:
q = "Summarize the abstract of the top retrieved paper."

out = ask_rag(q, k=5, max_chars=3500)

print("QUESTION:\n", out["question"])
print("\nANSWER:\n", out["answer"])
print("\nTOP RETRIEVED CHUNKS:")
display(out["retrieved_docs"])

QUESTION:
 Summarize the abstract of the top retrieved paper.

ANSWER:
 The top retrieved paper is [1] IDF revisited: A simple new derivation within the Robertson-Sp\"arck Jones probabilistic model. The abstract of this paper states that the authors show that a more intuitively plausible assumption suffices to justify the effectiveness of the inverse document frequency (IDF), whereas prior attempts took strong or complex assumptions as their starting point. Moreover, the new assumption provides a solution to an estimation problem that had been deemed intractable by Robertson and Walker (1997). [1]

TOP RETRIEVED CHUNKS:


,score,doc_id,chunk_id,title
0,0.306026,705.1161,705.1161__0,IDF revisited: A simple new derivation within ...
1,0.306026,704.3359,704.3359__1,Direct Optimization of Ranking Measures
2,0.261259,704.1676,704.1676__2,Personalizing Image Search Results on Flickr
3,0.256856,704.3359,704.3359__0,Direct Optimization of Ranking Measures
4,0.248356,704.1675,704.1675__0,Exploiting Social Annotation for Automatic Res...


In [41]:
question_bank = [
    "Summarize the abstract of the top retrieved paper.",
    "What is the main contribution of the retrieved paper?",
    "What methods are mentioned in the retrieved abstracts?",
    "What tasks or domains are discussed?",
    "List key terms from the retrieved papers.",
    "Compare the first two retrieved papers based on their abstracts.",
    "Which retrieved paper relates most to artificial intelligence and why?",
    "What categories (cs.LG, cs.AI, etc.) appear most often in the retrieved results?"
]

for q in question_bank:
    out = ask_rag(q, k=5, max_chars=3500)
    print("\n" + "="*90)
    print("Q:", q)
    print("A:", out["answer"])


Q: Summarize the abstract of the top retrieved paper.
A: The top retrieved paper is [4] Direct Optimization of Ranking Measures | 704.3359__0. The abstract states that Web page ranking and collaborative filtering require the optimization of sophisticated performance measures. Current Support Vector approaches are unable to optimize them directly and focus on pairwise comparisons instead. The paper presents a new approach which allows direct optimization of the relevant loss functions via structured estimation in Hilbert spaces. This is achieved by viewing the ranking problem as a linear assignment problem during training, which can be solved by the Hungarian Marriage algorithm. [4]

Q: What is the main contribution of the retrieved paper?
A: Based on the provided context, I can only find the main contribution of one paper. The main contribution of the paper "IDF revisited: A simple new derivation within the Robertson-Sparck Jones probabilistic model" [1] is that it shows that a more i

## Improvement — Include Categories in Context

To allow the model to answer meta-questions like:
"What categories (cs.LG, cs.AI, etc.) appear most often?"

We must include the `categories` field inside the context sent to the LLM.

In [43]:
def build_context(retrieved_df, max_chars: int = 3500):
    parts = []
    total = 0

    for i, row in enumerate(retrieved_df.itertuples(index=False), start=1):
        title = str(getattr(row, "title", ""))
        chunk_id = str(getattr(row, "chunk_id", ""))
        chunk_text = str(getattr(row, "chunk_text", ""))
        categories = str(getattr(row, "categories", ""))

        block = (
            f"[{i}] {title} | {chunk_id}\n"
            f"Categories: {categories}\n"
            f"{chunk_text}\n"
        )

        if total + len(block) > max_chars:
            break

        parts.append(block)
        total += len(block)

    return "\n---\n".join(parts)

## Compute Category Frequency (Deterministic, Not LLM-Based)

Instead of asking the model to count categories,
we compute category frequency directly from the retrieved results.
This is more accurate and reliable.

In [45]:
from collections import Counter

def compute_category_frequency(retrieved_df):
    all_categories = []

    for cats in retrieved_df["categories"].dropna():
        split_cats = cats.split()
        all_categories.extend(split_cats)

    counter = Counter(all_categories)
    return counter


# Example usage
q = "Summarize the abstract of the top retrieved paper."
retrieved = retrieve(q, k=5)

category_counts = compute_category_frequency(retrieved)

print("Category frequency in retrieved results:")
for cat, count in category_counts.most_common():
    print(f"{cat}: {count}")

Category frequency in retrieved results:
cs.IR: 4
cs.AI: 4
cs.CY: 2
cs.DL: 2
cs.CL: 1
cs.HC: 1


## Enhanced ask_rag — Include Category Analytics

We extend the RAG output to include:
- Retrieved documents
- Category frequency
- Final grounded answer

In [47]:
def ask_rag(question: str, k: int = 5, max_chars: int = 3500):
    retrieved = retrieve(question, k=k)
    context = build_context(retrieved, max_chars=max_chars)
    answer = generate_with_ollama(question, context)

    category_counts = compute_category_frequency(retrieved)

    return {
        "question": question,
        "answer": answer,
        "retrieved_docs": retrieved[["score", "doc_id", "chunk_id", "title", "categories"]],
        "category_counts": category_counts
    }

## Evaluation — Precision@K for Retrieval

To evaluate retrieval quality, we measure:

Precision@K = (Number of relevant retrieved documents) / K

We manually define which documents are relevant for a test query,
then compute how many of the top-K retrieved documents match.

This helps us measure whether FAISS retrieval is actually working.

In [49]:
def precision_at_k(retrieved_df, relevant_doc_ids, k=5):
    """
    retrieved_df: DataFrame from retrieve()
    relevant_doc_ids: set of doc_ids considered relevant
    k: top K documents to evaluate
    """
    top_k = retrieved_df.head(k)

    retrieved_ids = set(top_k["doc_id"].astype(str))
    relevant_ids = set(map(str, relevant_doc_ids))

    num_relevant = len(retrieved_ids & relevant_ids)

    return num_relevant / k

In [51]:
# Test Evaluation

# Example evaluation query
query = "What is preconditioned temporal difference learning?"

retrieved = retrieve(query, k=5)

# Manually define which doc_ids are relevant
relevant_doc_ids = {"704.1409"}  # You know this from inspection

score = precision_at_k(retrieved, relevant_doc_ids, k=5)

print("Precision@5:", score)
display(retrieved[["doc_id", "title"]])

Precision@5: 0.0


,doc_id,title
0,705.2318,Statistical Mechanics of Nonlinear On-line Lea...
1,706.1290,Temporal Reasoning without Transitive Tables
2,706.1290,Temporal Reasoning without Transitive Tables
3,705.2318,Statistical Mechanics of Nonlinear On-line Lea...
4,705.1999,A first-order Temporal Logic for Actions


## Evaluation Result — Precision@5 = 0.0

For the query:

> "What is preconditioned temporal difference learning?"

We manually defined the relevant document as:

- **doc_id: 704.1409**

However, the top-5 retrieved documents were:

- 705.2318 — Statistical Mechanics of Nonlinear On-line Learning  
- 706.1290 — Temporal Reasoning without Transitive Tables  
- 706.1290 — Temporal Reasoning without Transitive Tables  
- 705.2318 — Statistical Mechanics of Nonlinear On-line Learning  
- 705.1999 — A first-order Temporal Logic for Actions  

The expected relevant document (**704.1409**) did not appear in the top-5 results.

### Result

Precision@5 = 0.0

This means:

- 0 relevant documents were retrieved in the top 5
- Embedding-based retrieval did not rank the correct paper highly for this query

---

## Interpretation

This evaluation highlights a limitation of pure embedding-based retrieval:

- Semantically similar "temporal reasoning" papers were retrieved instead
- The specific concept "preconditioned temporal difference learning" was not strongly separated in embedding space
- Retrieval quality can degrade when multiple related topics share overlapping terminology

---

## Next Step

This motivates improvements such as:

- Increasing retrieval depth (higher K)
- Re-ranking with keyword overlap
- Hybrid retrieval (BM25 + embeddings)
- Category-aware boosting (e.g., cs.LG, cs.AI)

Evaluating retrieval quality using Precision@K helps systematically identify and improve weaknesses in the RAG pipeline.

## Mini Retrieval Evaluation (Precision@5)

To systematically evaluate retrieval quality, we define a small set of test queries.

For each query:
- We manually define the expected relevant `doc_id`
- Retrieve top-5 documents
- Compute Precision@5
- Record whether the relevant document appears in top-5 (Hit@5)

This creates a lightweight evaluation benchmark for the RAG system.

In [54]:
# Define evaluation set (manually curated)

evaluation_set = [
    {
        "query": "What is preconditioned temporal difference learning?",
        "relevant_doc_ids": {"704.1409"}
    },
    {
        "query": "What is temporal reasoning without transitive tables?",
        "relevant_doc_ids": {"706.1290"}
    },
    {
        "query": "What is statistical mechanics of nonlinear online learning?",
        "relevant_doc_ids": {"705.2318"}
    },
    {
        "query": "What is a first-order temporal logic for actions?",
        "relevant_doc_ids": {"705.1999"}
    }
]

In [56]:
import pandas as pd

results = []

for item in evaluation_set:
    query = item["query"]
    relevant_ids = item["relevant_doc_ids"]
    
    retrieved = retrieve(query, k=5)
    
    p_at_5 = precision_at_k(retrieved, relevant_ids, k=5)
    
    retrieved_ids = set(retrieved["doc_id"].astype(str))
    hit_at_5 = int(len(retrieved_ids & relevant_ids) > 0)
    
    results.append({
        "query": query,
        "expected_doc_id": list(relevant_ids)[0],
        "precision@5": p_at_5,
        "hit@5": hit_at_5
    })

evaluation_df = pd.DataFrame(results)

evaluation_df

,query,expected_doc_id,precision@5,hit@5
0,What is preconditioned temporal difference lea...,704.1409,0.0,0
1,What is temporal reasoning without transitive ...,706.1290,0.0,0
2,What is statistical mechanics of nonlinear onl...,705.2318,0.2,1
3,What is a first-order temporal logic for actions?,705.1999,0.2,1


In [58]:
print("Average Precision@5:", evaluation_df["precision@5"].mean())
print("Hit Rate@5:", evaluation_df["hit@5"].mean())

Average Precision@5: 0.1
Hit Rate@5: 0.5


## Mini Evaluation Results

We evaluated retrieval quality using a small manually curated query set.

Metrics computed:

- **Precision@5** — proportion of relevant documents in top 5
- **Hit@5** — whether at least one relevant document appears in top 5

### Results Summary

- **Average Precision@5:** 0.10  
- **Hit Rate@5:** 0.50  

This means:

- On average, 10% of the top-5 retrieved documents were relevant.
- For 50% of evaluation queries, at least one relevant document appeared in the top-5 results.

---

## Interpretation

The results indicate:

- Embedding-based retrieval is partially effective.
- Some domain-specific queries (e.g., highly specific concepts) are not ranked correctly.
- Semantically similar but non-target documents are often retrieved instead.

This evaluation highlights the need for:

- Improved ranking (re-ranking or hybrid retrieval)
- Metadata-aware boosting (e.g., category filtering)
- Further tuning of chunking or embeddings

---

## Key Insight

Measuring retrieval performance with Precision@K and Hit@K provides a structured way to:

- Diagnose retrieval weaknesses
- Compare improvements objectively
- Iteratively refine the RAG system

This moves the system from a demo-level pipeline to an evaluation-driven AI search system.

# 04 — RAG Query Pipeline

## Overview

This notebook implements the complete Retrieval-Augmented Generation (RAG) query pipeline for semantic search over research papers.

The system connects:

- FAISS-based semantic retrieval  
- Context construction from retrieved chunks  
- Local LLM generation via Ollama  
- Citation-grounded responses  
- Retrieval evaluation using Precision@5 and Hit@5  

---

## Pipeline Flow

1. **Retrieve**
   - Embed user query
   - Retrieve top-K relevant chunks from FAISS

2. **Build Context**
   - Combine chunk text with metadata (title, chunk_id, categories)

3. **Generate**
   - Send structured context + query to LLM
   - Enforce grounded answers with citations

4. **Evaluate**
   - Compute category frequency (deterministic)
   - Measure retrieval performance using Precision@5 and Hit@5

---

## Key Outcome

The notebook completes an end-to-end, evaluation-driven RAG workflow:

- Semantic retrieval  
- Grounded answer generation  
- Metadata-aware analysis  
- Quantitative retrieval evaluation  

This mirrors real-world AI search system design and emphasizes measurable performance over demo-only outputs.